# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Health Score and Random Forest feature importance

The paper defines a 100-point Health Score using impressions (30 points), average position (30 points), CTR (20 points), and scroll depth (20 points). It then reports that a Random Forest trained to predict this score assigns 43% importance to average position, 32% to impressions, 15% to scroll depth, and 8% to CTR — 98% combined across the four ingredients of the score.

**Methodology question:**  
Because these four variables are already used to construct the Health Score, to what extent should the Random Forest feature-importance result be interpreted as a new finding rather than the model recovering the predefined scoring recipe? The result is useful as a check of the model, but I would be cautious about treating it as independent evidence of what drives SEO health.

### Finding 2 — 71% holdout accuracy

The paper reports 71% holdout accuracy for a Logistic Regression model used to distinguish growing and declining pages. The methodology describes an 80/20 split for the exploratory machine-learning analyses.

**Methodology question:**  
What exactly was kept separate when the 80/20 split was created? In particular, were observations from the same client or related time periods allowed to appear in both training and holdout data? A clearer split boundary would help determine whether the reported holdout accuracy reflects performance on genuinely unseen data.


In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### W05 result — Before

In Week 5, I trained a Random Forest using seven features:

- `imp_prev30`
- `clk_last30`
- `pos_last30`
- `visible_queries`
- `rare_share`
- `anon_share`
- `top_query_share`

The model was evaluated using a client-grouped split.

The measured Week-5 performance was:

- Accuracy: 81.57%
- ROC-AUC: 0.865

I keep this result unchanged as the before-validation reference.

In [3]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [5]:
# Build the earlier training snapshot
TRAIN_END = "2026-04-30"

train_snapshot = con.sql(f"""
    WITH windowed AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TRAIN_END}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 60 DAY
                     AND report_date <= DATE '{TRAIN_END}' - INTERVAL 30 DAY
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TRAIN_END}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN report_date > DATE '{TRAIN_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TRAIN_END}'
                    THEN gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']}

        WHERE report_date <= DATE '{TRAIN_END}'
          AND report_date > DATE '{TRAIN_END}' - INTERVAL 60 DAY

        GROUP BY client_hash_id, content_hash_id
    )

    SELECT *
    FROM windowed
    WHERE imp_prev30 >= 100
""").df()

# Same Week-5 label
train_snapshot["is_declining"] = (
    train_snapshot["imp_last30"]
    < 0.8 * train_snapshot["imp_prev30"]
).astype(int)

print(train_snapshot.shape)
train_snapshot.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(100849, 7)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,is_declining
0,client_62f4a7e64f5e0096,content_5d562ebeda84bdc7,312.0,773.0,0.0,15.507531,1
1,client_62f4a7e64f5e0096,content_d6f555d072e070be,13273.0,25534.0,22.0,5.801563,1
2,client_62f4a7e64f5e0096,content_dabbfdf80d8773f0,1089.0,1448.0,6.0,5.627420,1
3,client_62f4a7e64f5e0096,content_a761d62e362213d3,3493.0,5595.0,5.0,5.159570,1
4,client_62f4a7e64f5e0096,content_eb6738715ef9bac1,167.0,412.0,0.0,23.942394,1


In [6]:
TEST_END = "2026-06-30"

test_snapshot = con.sql(f"""
    WITH windowed AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TEST_END}'
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_last30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 60 DAY
                     AND report_date <= DATE '{TEST_END}' - INTERVAL 30 DAY
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS imp_prev30,

            SUM(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TEST_END}'
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS clk_last30,

            AVG(
                CASE
                    WHEN report_date > DATE '{TEST_END}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{TEST_END}'
                    THEN gsc_avg_position
                END
            ) AS pos_last30

        FROM {TABLES['fact_daily']}

        WHERE report_date <= DATE '{TEST_END}'
          AND report_date > DATE '{TEST_END}' - INTERVAL 60 DAY

        GROUP BY client_hash_id, content_hash_id
    )

    SELECT *
    FROM windowed
    WHERE imp_prev30 >= 100
""").df()

test_snapshot["is_declining"] = (
    test_snapshot["imp_last30"]
    < 0.8 * test_snapshot["imp_prev30"]
).astype(int)

print(test_snapshot.shape)
test_snapshot.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(111247, 7)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,is_declining
0,client_3ffa76342f366962,content_bdc656fc8f037ac0,124.0,173.0,7.0,4.931609,1
1,client_3ffa76342f366962,content_315f75aa07a662bf,48.0,139.0,0.0,2.361667,1
2,client_e547b89c05043229,content_ded3d63f83e7a4cf,1795.0,588.0,4.0,9.341574,0
3,client_e547b89c05043229,content_b3a828afc221c27a,85.0,115.0,0.0,13.640051,1
4,client_e547b89c05043229,content_0252039a1f263e4e,299.0,352.0,0.0,34.854224,0


In [7]:
features_w06 = [
    "imp_prev30",
    "clk_last30",
    "pos_last30"
]

X_train = train_snapshot[features_w06]
y_train = train_snapshot["is_declining"]

X_test = test_snapshot[features_w06]
y_test = test_snapshot["is_declining"]

In [8]:
from sklearn.ensemble import RandomForestClassifier

model_w06 = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_w06.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [9]:
y_pred = model_w06.predict(X_test)
y_prob = model_w06.predict_proba(X_test)[:, 1]

In [10]:
from sklearn.metrics import accuracy_score, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("W06 Time-aware Accuracy:", accuracy)
print("W06 Time-aware ROC-AUC:", auc)
print("Test positive rate:", y_test.mean())

W06 Time-aware Accuracy: 0.6074321105288233
W06 Time-aware ROC-AUC: 0.7092777080638085
Test positive rate: 0.6431364441288304


### Before vs After: Validation Comparison

| Evaluation | Feature set | Validation design | Accuracy | ROC-AUC |
|---|---|---|---:|---:|
| W05 — Before | 7 features | Client-grouped split | 81.57% | 0.865 |
| W06 — After | 3 temporally safe features | Time-aware (Apr 30 → Jun 30) | 60.85% | 0.709 |

The stricter time-aware evaluation measured lower performance than the Week-5
evaluation. However, the difference cannot be attributed entirely to the
validation design because the Week-6 audit uses three temporally safe features
instead of the seven features used in Week 5.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I checked the Week-5 features against the label definition and the
prediction timeline. I excluded features that were label-derived or
could not be historically aligned to the April 30 training snapshot.

### 3.1 Label-derived feature check

The Week-5 label `is_declining` is defined as:

`imp_last30 < 0.8 × imp_prev30`

Therefore, `imp_last30` is directly involved in constructing the label and
must not be used as a model feature.

For the Week-6 time-aware audit, I used only:

- `imp_prev30`
- `clk_last30`
- `pos_last30`

`imp_last30` was retained only for constructing the evaluation label.

In [11]:
features_w06

['imp_prev30', 'clk_last30', 'pos_last30']

### 3.2 Future and overlapping window check

For the April 30 training snapshot, all performance features were calculated
using data available on or before April 30, 2026.

The training windows were:

- `imp_last30`: April 1–April 30, used only to construct the label
- `imp_prev30`: March 2–March 31, used as a model feature
- `clk_last30`: April 1–April 30
- `pos_last30`: April 1–April 30

The model does not use `imp_last30` as a feature because it is part of the
label definition.

For the June 30 test snapshot, the same window logic was applied using data
available through June 30, 2026.

### 3.3 Query-feature temporal alignment audit

In Week 5, the query-derived features were created from
`fact_content_query_90d`:

- `visible_queries`
- `rare_share`
- `anon_share`
- `top_query_share`

I checked the available query table and found that its 90-day window has:

- `window_start = 2026-04-02`
- `window_end = 2026-06-30`

The Week-6 training snapshot is anchored at April 30, 2026.

Therefore, the available query table extends beyond the April 30 training
cutoff. It cannot provide a historically aligned query feature set for the
April training snapshot.

Using these query features for the April training snapshot could expose the
model to information from after the training cutoff. I therefore excluded
the query-derived features from the Week-6 time-aware audit.

This is a temporal-alignment concern, so the Week-6 model uses only the three
performance features that can be constructed from data available at the
corresponding snapshot date.

In [12]:
print("Query table window:")
print("window_start:", "2026-04-02")
print("window_end  :", "2026-06-30")

print("\nW06 query features excluded:")
print([
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
])

print("\nW06 features retained:")
print(features_w06)

Query table window:
window_start: 2026-04-02
window_end  : 2026-06-30

W06 query features excluded:
['visible_queries', 'rare_share', 'anon_share', 'top_query_share']

W06 features retained:
['imp_prev30', 'clk_last30', 'pos_last30']


### 3.4 Leakage audit conclusion

The audit found that `imp_last30` is label-derived and was therefore
excluded from the model features.

I also checked the prediction timeline and constructed the April 30
training features only from information available by that cutoff.

The Week-5 query-derived features were not used in the Week-6 time-aware
audit because the available query table has a window ending on June 30,
which cannot be historically aligned to the April 30 training cutoff.

The final Week-6 feature set was therefore:

- `imp_prev30`
- `clk_last30`
- `pos_last30`

This produces a more conservative validation setup and avoids relying on
features whose historical timing cannot be verified.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original W05 claim

Overall, the model captures the main decline patterns while remaining
suitable as a decision-support tool rather than a final decision maker.

### Why I am revising this claim

The Week-5 model measured 81.57% accuracy under the Week-5 validation
design. However, the Week-6 time-aware audit measured 60.85% accuracy
and a ROC-AUC of 0.709 using three temporally safe features.

Therefore, the original wording "captures the main decline patterns" is
stronger than the evidence from the stricter time-aware evaluation supports.

### Rewritten claim

The Week-5 model showed measured predictive signal under its tested
client-grouped evaluation. Under the Week-6 time-aware audit, the model
retained directional ranking signal with a ROC-AUC of 0.709, although its
60.85% accuracy did not exceed the 64.31% majority-class baseline.

The model is therefore better described as a decision-support signal for
ranking potentially declining content, rather than as a reliable predictor
of future decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.